## Helpers

Contains the `HELPERS` notebook section.


In [ ]:
# HELPERS
PROFILE_EXCLUDE = {
    "x",
    "s_coord",
    "a",
    "muK",
    "T_profile",
    "closure_error",
    "message",
    "branch_label",
    "coordinate",
    "bvp_message",
}
PASS_REL_TOL = 1.0e-6
NEAR_REL_TOL = 1.0e-4


def _safe_float(value, default=np.nan):
    try:
        value = float(np.asarray(value).reshape(()))
    except Exception:
        return float(default)
    return value if np.isfinite(value) else float(default)


def _safe_eval(func, x):
    try:
        value = float(func(float(x)))
    except Exception:
        return np.nan
    return value if np.isfinite(value) else np.nan


def find_bracketed_root(func, grid, prefer=None, xtol=1.0e-6):
    grid = np.asarray(grid, dtype=float)
    values = np.array([_safe_eval(func, x) for x in grid], dtype=float)
    finite = np.isfinite(values)
    grid = grid[finite]
    values = values[finite]
    if grid.size < 2:
        raise RuntimeError("not enough finite samples to bracket a root")
    exact = np.where(np.isclose(values, 0.0, atol=1.0e-9, rtol=0.0))[0]
    if exact.size:
        return float(grid[int(exact[0])])
    sign_changes = np.where(values[:-1] * values[1:] < 0.0)[0]
    if sign_changes.size == 0:
        finite_msg = f"finite samples={grid.size}, value range=[{np.nanmin(values):.6g}, {np.nanmax(values):.6g}]"
        raise RuntimeError("no sign change found; " + finite_msg)
    if prefer is None:
        idx = int(sign_changes[0])
    else:
        mids = 0.5 * (grid[sign_changes] + grid[sign_changes + 1])
        idx = int(sign_changes[np.argmin(np.abs(mids - float(prefer)))])
    return float(brentq(func, float(grid[idx]), float(grid[idx + 1]), xtol=xtol, rtol=1.0e-8, maxiter=200))


_CRITICAL_CACHE = {}


def critical_state_for_temperature(T):
    key = round(float(T), 12)
    if key in _CRITICAL_CACHE:
        return dict(_CRITICAL_CACHE[key])

    mu_grid = np.linspace(850.0, 1800.0, 381)

    def pressure_difference(muB):
        return (
            phase_velocity.PNM(muB, T, param=param, NM_type=NM_type)
            - phase_velocity.PQM(muB, 0.0, B14, T, ms=ms)
        )

    muB_crit = find_bracketed_root(pressure_difference, mu_grid, prefer=1150.0)
    P_crit = float(phase_velocity.PQM(muB_crit, 0.0, B14, T, ms=ms))
    nB_crit = float(phase_velocity.nB_NM(muB_crit, T, param=param, NM_type=NM_type))
    out = {
        "T_MeV": float(T),
        "muB_crit_MeV": muB_crit,
        "P_crit_MeV4": P_crit,
        "nB_crit": nB_crit,
        "nB_crit_over_n0": nB_crit / n0,
    }
    _CRITICAL_CACHE[key] = dict(out)
    return out


def trial_id(T, metastability):
    T_label = f"T{float(T):.6g}".replace("-", "m").replace(".", "p")
    M_label = f"M{float(metastability):.6g}".replace("-", "m").replace(".", "p")
    return f"{T_label}__{M_label}"


def save_result_npz(path, result, metadata):
    payload = {}
    for key, value in metadata.items():
        payload[f"meta_{key}"] = np.array(value)
    for key, value in result.items():
        if key.startswith("_"):
            continue
        if isinstance(value, str):
            payload[key] = np.array(value)
        elif isinstance(value, (int, float, bool, np.integer, np.floating, np.bool_)):
            payload[key] = np.array(value)
        else:
            arr = np.asarray(value)
            if arr.dtype.kind in "biufcUS":
                payload[key] = arr
    np.savez(path, **payload)


def numeric_profile_arrays(result):
    if "a" not in result:
        return {}
    a = np.asarray(result["a"], dtype=float)
    if a.ndim != 1 or a.size < 2:
        return {}
    n = a.size
    profiles = {}
    for key, value in result.items():
        if key in PROFILE_EXCLUDE or key.startswith("_"):
            continue
        try:
            arr = np.asarray(value, dtype=float)
        except Exception:
            continue
        if arr.ndim == 1 and arr.size == n and np.any(np.isfinite(arr)):
            profiles[key] = arr
    if all(key in result for key in ("h_profile", "u", "r_gamma_profile")):
        profiles["E_reconstructed"] = (
            np.asarray(result["h_profile"], dtype=float)
            * np.asarray(result["u"], dtype=float)
            * np.asarray(result["r_gamma_profile"], dtype=float)
        )
    return profiles


def constancy_metric(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size < 2:
        return {
            "n_finite": int(finite.size),
            "min": np.nan,
            "max": np.nan,
            "abs_range": np.nan,
            "rel_range": np.nan,
            "label": "insufficient",
        }
    abs_range = float(np.nanmax(finite) - np.nanmin(finite))
    scale = float(max(np.nanmax(np.abs(finite)), np.finfo(float).tiny))
    rel_range = float(abs_range / scale)
    if rel_range <= PASS_REL_TOL:
        label = "pass"
    elif rel_range <= NEAR_REL_TOL:
        label = "near"
    else:
        label = "fail"
    return {
        "n_finite": int(finite.size),
        "min": float(np.nanmin(finite)),
        "max": float(np.nanmax(finite)),
        "abs_range": abs_range,
        "rel_range": rel_range,
        "label": label,
    }


def make_summary_row(T, metastability, crit, result, elapsed_s, npz_path, seed_note):
    return {
        "trial_id": trial_id(T, metastability),
        "T_MeV": float(T),
        "metastability": float(metastability),
        "B14_MeV": float(B14),
        "aQstar_input": float(aQstar),
        "nB_crit_over_n0": float(crit["nB_crit_over_n0"]),
        "nB_N_over_n0": float((crit["nB_crit"] + float(metastability) * n0) / n0),
        "nB_crit": float(crit["nB_crit"]),
        "nB_N": float(crit["nB_crit"] + float(metastability) * n0),
        "muB_crit_MeV": float(crit["muB_crit_MeV"]),
        "P_crit_MeV4": float(crit["P_crit_MeV4"]),
        "success": bool(result.get("success", False)),
        "message": str(result.get("message", "")),
        "jB": _safe_float(result.get("jB")),
        "u_N_m_s": _safe_float(result.get("u_N")) * const.m_per_sec,
        "T_Q_MeV": _safe_float(result.get("T_Q")),
        "T_Qstar_MeV": _safe_float(result.get("T_Qstar")),
        "tail_residual_norm": _safe_float(result.get("tail_residual_norm")),
        "E_right_residual_norm": _safe_float(result.get("E_right_residual_norm")),
        "bvp_status": int(result.get("bvp_status", -999)) if np.isfinite(_safe_float(result.get("bvp_status"), -999)) else -999,
        "bvp_nodes": int(result.get("bvp_nodes", 0)) if np.isfinite(_safe_float(result.get("bvp_nodes"), 0)) else 0,
        "continuation_used": bool(result.get("continuation_used", False)),
        "continuation_steps": int(result.get("continuation_steps", 0)) if np.isfinite(_safe_float(result.get("continuation_steps"), 0)) else 0,
        "elapsed_s": float(elapsed_s),
        "npz_path": str(npz_path),
        "seed_note": str(seed_note),
    }


def append_csv(path, rows):
    df = pd.DataFrame(rows)
    if not path.exists():
        df.to_csv(path, index=False)
    else:
        df.to_csv(path, mode="a", index=False, header=False)
    return df

print("Helper functions loaded.")

## Run 4X4 Scan

Contains the `RUN_4X4_SCAN` notebook section.


In [ ]:
# RUN_4X4_SCAN
# Re-running this cell appends a fresh run unless force_rerun is False and a successful raw NPZ exists.
force_rerun = False

trial_rows = []
metric_rows = []
last_success_by_T = {}
last_success_global = None

if trial_summary_path.exists():
    previous_summary = pd.read_csv(trial_summary_path)
else:
    previous_summary = pd.DataFrame()

for i_T, T in enumerate(temperature_values):
    crit = critical_state_for_temperature(float(T))
    print(
        f"Critical state T={float(T):g} MeV: "
        f"nB_crit/n0={crit['nB_crit_over_n0']:.6g}, muB_crit={crit['muB_crit_MeV']:.6g} MeV"
    )
    for i_M, metastability in enumerate(metastability_values):
        tid = trial_id(T, metastability)
        npz_path = raw_dir / f"{tid}.npz"
        if (
            not force_rerun
            and npz_path.exists()
            and not previous_summary.empty
            and bool(previous_summary.loc[previous_summary["trial_id"] == tid, "success"].tail(1).any())
        ):
            print(f"Skipping existing successful trial {tid}")
            continue

        nB_N = crit["nB_crit"] + float(metastability) * n0
        seed_jB = None
        seed_TQ = None
        seed_note = "default"
        if float(T) in last_success_by_T:
            seed_jB = last_success_by_T[float(T)]["jB"]
            seed_TQ = last_success_by_T[float(T)]["T_Q"]
            seed_note = "previous success at same T"
        elif last_success_global is not None:
            seed_jB = last_success_global["jB"]
            seed_TQ = last_success_global["T_Q"]
            seed_note = "previous global success"
        if seed_jB is None or not np.isfinite(seed_jB) or seed_jB <= 0:
            seed_jB = None
        if seed_TQ is None or not np.isfinite(seed_TQ) or seed_TQ <= 0:
            seed_TQ = 60.0
            if seed_note == "default":
                seed_note = "default TQ=60"

        print(
            f"Running {tid}: T={float(T):g} MeV, M={float(metastability):g}, "
            f"nB_N/n0={nB_N/n0:.6g}, seed={seed_note}"
        )
        t0 = time.perf_counter()
        try:
            result = phase_velocity.solve_front_energy_conserving(
                T=float(T),
                nB_N=float(nB_N),
                B_one_forth=float(B14),
                aQstar=float(aQstar),
                ms=float(ms),
                param=param,
                NM_type=NM_type,
                jB_guess=seed_jB,
                TQ_guess=seed_TQ,
                **solver_controls,
            )
        except Exception as exc:
            result = {
                "success": False,
                "message": f"exception: {exc}",
                "jB": np.nan,
                "T_Q": np.nan,
                "T_Qstar": np.nan,
                "tail_residual_norm": np.nan,
                "E_right_residual_norm": np.nan,
                "bvp_status": -999,
                "bvp_nodes": 0,
            }
        elapsed_s = time.perf_counter() - t0

        metadata = {
            "trial_id": tid,
            "T_MeV": float(T),
            "metastability": float(metastability),
            "nB_N": float(nB_N),
            "nB_crit": float(crit["nB_crit"]),
            "B14_MeV": float(B14),
            "aQstar_input": float(aQstar),
        }
        save_result_npz(npz_path, result, metadata)
        row = make_summary_row(T, metastability, crit, result, elapsed_s, npz_path, seed_note)
        trial_rows.append(row)
        append_csv(trial_summary_path, [row])

        if bool(result.get("success", False)):
            last_success_by_T[float(T)] = {"jB": float(result["jB"]), "T_Q": float(result["T_Q"])}
            last_success_global = {"jB": float(result["jB"]), "T_Q": float(result["T_Q"])}
            profiles = numeric_profile_arrays(result)
            rows_this_trial = []
            for variable, values in profiles.items():
                metric = constancy_metric(values)
                metric_row = {
                    "trial_id": tid,
                    "T_MeV": float(T),
                    "metastability": float(metastability),
                    "variable": variable,
                    **metric,
                }
                rows_this_trial.append(metric_row)
                metric_rows.append(metric_row)
            if rows_this_trial:
                append_csv(profile_metrics_path, rows_this_trial)
        print(
            f"  success={bool(result.get('success', False))}, "
            f"jB={_safe_float(result.get('jB')):.6g}, T_Q={_safe_float(result.get('T_Q')):.6g}, "
            f"elapsed={elapsed_s:.1f}s, message={str(result.get('message', ''))[:100]}"
        )

trial_summary_df = pd.read_csv(trial_summary_path) if trial_summary_path.exists() else pd.DataFrame(trial_rows)
profile_metrics_df = pd.read_csv(profile_metrics_path) if profile_metrics_path.exists() else pd.DataFrame(metric_rows)

latest_trials = (
    trial_summary_df.drop_duplicates(subset=["trial_id"], keep="last")
    if not trial_summary_df.empty else trial_summary_df
)
latest_metrics = (
    profile_metrics_df.drop_duplicates(subset=["trial_id", "variable"], keep="last")
    if not profile_metrics_df.empty else profile_metrics_df
)

if not latest_metrics.empty:
    matrix_df = latest_metrics.pivot_table(index="variable", columns="trial_id", values="label", aggfunc="last")
    matrix_df.to_csv(matrix_path)
else:
    matrix_df = pd.DataFrame()

print(f"Wrote trial summary: {trial_summary_path}")
print(f"Wrote profile metrics: {profile_metrics_path}")
print(f"Wrote pass/fail matrix: {matrix_path}")
display(latest_trials)
if not latest_metrics.empty:
    display(latest_metrics.head(20))

## Results Visualization

Contains the `RESULTS_VISUALIZATION` notebook section.


In [ ]:
# RESULTS_VISUALIZATION
if not trial_summary_path.exists():
    raise RuntimeError("Run RUN_4X4_SCAN first.")

trial_summary_df = pd.read_csv(trial_summary_path).drop_duplicates(subset=["trial_id"], keep="last")
profile_metrics_df = (
    pd.read_csv(profile_metrics_path).drop_duplicates(subset=["trial_id", "variable"], keep="last")
    if profile_metrics_path.exists()
    else pd.DataFrame()
)

success_grid = trial_summary_df.pivot_table(index="T_MeV", columns="metastability", values="success", aggfunc="last")
success_grid = success_grid.reindex(index=temperature_values, columns=metastability_values)

display(Markdown("### Trial success grid"))
display(success_grid)

display(Markdown("### Latest trial summary"))
display(trial_summary_df.sort_values(["T_MeV", "metastability"]))

if profile_metrics_df.empty:
    display(Markdown("No profile metrics were produced because no successful trials returned profile arrays."))
else:
    label_counts = (
        profile_metrics_df.groupby(["variable", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["pass", "near", "fail", "insufficient"], fill_value=0)
    )
    worst_ranges = (
        profile_metrics_df.groupby("variable", as_index=True)["rel_range"]
        .max()
        .sort_values(ascending=False)
        .rename("worst_rel_range")
    )
    variable_summary = label_counts.join(worst_ranges)

    display(Markdown("### Variable-level constancy summary"))
    display(variable_summary)

    heatmap_data = profile_metrics_df.pivot_table(
        index="variable",
        columns="trial_id",
        values="rel_range",
        aggfunc="last",
    )
    heatmap_log = np.log10(heatmap_data.astype(float).clip(lower=np.finfo(float).tiny))

    fig, ax = plt.subplots(figsize=(max(10.0, 0.55 * heatmap_log.shape[1]), max(4.5, 0.35 * heatmap_log.shape[0])))
    im = ax.imshow(heatmap_log.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_yticks(np.arange(heatmap_log.shape[0]))
    ax.set_yticklabels(heatmap_log.index)
    ax.set_xticks(np.arange(heatmap_log.shape[1]))
    ax.set_xticklabels(heatmap_log.columns, rotation=60, ha="right")
    ax.set_title(r"$\log_{10}$ relative range by profile variable")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$\log_{10}(\Delta y / \max|y|)$")
    fig.tight_layout()
    plt.show()

    display(Markdown("### Worst relative range per variable"))
    display(worst_ranges.to_frame())

## Summary

Contains the `SUMMARY` notebook section.


In [ ]:
# SUMMARY
if not trial_summary_path.exists():
    raise RuntimeError("Run RUN_4X4_SCAN first.")

trial_summary_df = pd.read_csv(trial_summary_path).drop_duplicates(subset=["trial_id"], keep="last")
profile_metrics_df = (
    pd.read_csv(profile_metrics_path).drop_duplicates(subset=["trial_id", "variable"], keep="last")
    if profile_metrics_path.exists()
    else pd.DataFrame()
)

n_expected = len(temperature_values) * len(metastability_values)
n_trials = len(trial_summary_df)
n_success = int(trial_summary_df["success"].astype(bool).sum()) if n_trials else 0
n_failed = n_trials - n_success

summary_lines = [
    "# 26SU-3 result summary",
    "",
    f"Ran {n_trials}/{n_expected} planned temperature/metastability trials.",
    f"Successful solver profiles: {n_success}; failed solver profiles: {n_failed}.",
]

if n_failed:
    failed_bits = []
    for _, row in trial_summary_df.loc[~trial_summary_df["success"].astype(bool)].iterrows():
        failed_bits.append(
            f"T={row['T_MeV']:g} MeV, M={row['metastability']:g}: {str(row['message'])[:140]}"
        )
    summary_lines.extend(["", "Failed points:", *[f"- {bit}" for bit in failed_bits]])

if profile_metrics_df.empty:
    summary_lines.extend(["", "No successful profile metrics were available to assess piecewise constancy."])
else:
    label_counts = (
        profile_metrics_df.groupby(["variable", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["pass", "near", "fail", "insufficient"], fill_value=0)
    )
    worst_ranges = profile_metrics_df.groupby("variable")["rel_range"].max().sort_values(ascending=False)
    all_pass_variables = label_counts.index[(label_counts.get("fail", 0) == 0) & (label_counts.get("near", 0) == 0)].tolist()
    any_fail_variables = label_counts.index[label_counts.get("fail", 0) > 0].tolist()
    near_variables = label_counts.index[(label_counts.get("near", 0) > 0) & (label_counts.get("fail", 0) == 0)].tolist()

    summary_lines.extend([
        "",
        f"Variables passing the strict {PASS_REL_TOL:g} relative-range test in every successful trial: "
        + (", ".join(all_pass_variables) if all_pass_variables else "none"),
        f"Variables with at least one non-constant failure above {NEAR_REL_TOL:g}: "
        + (", ".join(any_fail_variables) if any_fail_variables else "none"),
        f"Variables only in the near band ({PASS_REL_TOL:g}, {NEAR_REL_TOL:g}]: "
        + (", ".join(near_variables) if near_variables else "none"),
        "",
        "Worst relative ranges:",
    ])
    for variable, value in worst_ranges.items():
        summary_lines.append(f"- {variable}: {value:.3e}")

    if "E_reconstructed" in worst_ranges.index:
        summary_lines.extend([
            "",
            "The reconstructed conserved energy flux E = h_profile * u * r_gamma_profile "
            f"has worst relative range {worst_ranges.loc['E_reconstructed']:.3e} across successful trials.",
        ])

summary_text = "\n".join(summary_lines)
summary_path = output_dir / "result_summary.md"
summary_path.write_text(summary_text + "\n", encoding="utf-8")
display(Markdown(summary_text))
print(f"Wrote summary: {summary_path}")

## Charge-neutral quark-only EOS comparison

This section compares the current charge-neutral quark-only `PQM` implementation against the previous non-neutral chemical-potential assignment. The old EOS is reconstructed locally for testing and plotting; it is not imported from the package.


In [ ]:
# CHARGE_NEUTRAL_PQM_CALCULATION
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from RMFsolver import SQMsolver as sqm

# Defaults are deliberately local so this comparison can be rerun independently.
compare_muB = float(globals().get("compare_muB", 1100.0))
compare_T = float(globals().get("compare_T", 10.0))
compare_B14 = float(globals().get("compare_B14", globals().get("B14", 180.0)))
compare_ms = float(globals().get("compare_ms", globals().get("ms", 0.0)))
compare_upB = float(globals().get("compare_upB", 5000.0))
compare_muK_grid = np.linspace(0.0, 420.0, 121)


def _display(obj):
    try:
        display(obj)
    except NameError:
        print(obj)


def old_quark_mu_triplet(muB, muK):
    """Old non-neutral assignment used before the auxiliary muQ solve."""
    return (
        float(muB / 3.0),
        float(muB / 3.0 + muK / 2.0),
        float(muB / 3.0 - muK / 2.0),
    )


def old_quark_state(muB, muK, T, ms=0.0, upB=5000.0):
    mu_u, mu_d, mu_s = old_quark_mu_triplet(muB, muK)
    state = sqm._quark_uds_state_from_triplet(mu_u, mu_d, mu_s, T, ms, upB=upB)
    state = dict(state)
    state["chemical_potentials"] = {"mu_u": mu_u, "mu_d": mu_d, "mu_s": mu_s, "mu_Q": 0.0}
    return state


def old_PQM(muB, muK, B_one_forth, T, ms=0.0, upB=5000.0):
    # This intentionally mirrors the old package behavior: quark pressure minus bag pressure.
    # It does not include gluon pressure and does not enforce electric charge neutrality.
    return float(old_quark_state(muB, muK, T, ms=ms, upB=upB)["pressure"] - B_one_forth**4)


def new_quark_state(muB, muK, T, ms=0.0, upB=5000.0):
    mu_u, mu_d, mu_s = sqm._quark_mu_triplet_neutral(muB, muK, T, ms=ms, upB=upB)
    state = sqm._quark_uds_state(muB, muK, T, ms=ms, upB=upB)
    mu_Q = 1.5 * (mu_u - muB / 3.0)
    state = dict(state)
    state["chemical_potentials"] = dict(state["chemical_potentials"])
    state["chemical_potentials"]["mu_Q"] = float(mu_Q)
    return state


def charge_residual(species):
    # Physical color-included quark densities: n_i = 3 * n_f,colorless.
    return float(2.0 * species["n_u"] - species["n_d"] - species["n_s"])


def normalized_charge_residual(state):
    nB = float(state["nB"])
    return charge_residual(state["species"]) / max(3.0 * abs(nB), 1.0)


rows = []
for muK in compare_muK_grid:
    old_state = old_quark_state(compare_muB, muK, compare_T, ms=compare_ms, upB=compare_upB)
    new_state = new_quark_state(compare_muB, muK, compare_T, ms=compare_ms, upB=compare_upB)
    old_P = old_PQM(compare_muB, muK, compare_B14, compare_T, ms=compare_ms, upB=compare_upB)
    new_P = sqm.PQM(compare_muB, muK, compare_B14, compare_T, ms=compare_ms, upB=compare_upB)
    rows.append(
        {
            "muB_MeV": compare_muB,
            "muK_MeV": float(muK),
            "T_MeV": compare_T,
            "B14_MeV": compare_B14,
            "ms_MeV": compare_ms,
            "P_old_MeV4": old_P,
            "P_new_MeV4": new_P,
            "delta_P_MeV4": new_P - old_P,
            "relative_delta_P": (new_P - old_P) / max(abs(old_P), 1.0),
            "old_charge_over_3nB": normalized_charge_residual(old_state),
            "new_charge_over_3nB": normalized_charge_residual(new_state),
            "muQ_new_MeV": new_state["chemical_potentials"]["mu_Q"],
            "old_mu_u_MeV": old_state["chemical_potentials"]["mu_u"],
            "old_mu_d_MeV": old_state["chemical_potentials"]["mu_d"],
            "old_mu_s_MeV": old_state["chemical_potentials"]["mu_s"],
            "new_mu_u_MeV": new_state["chemical_potentials"]["mu_u"],
            "new_mu_d_MeV": new_state["chemical_potentials"]["mu_d"],
            "new_mu_s_MeV": new_state["chemical_potentials"]["mu_s"],
            "old_nu_over_nB": old_state["species"]["n_u"] / old_state["nB"],
            "old_nd_over_nB": old_state["species"]["n_d"] / old_state["nB"],
            "old_ns_over_nB": old_state["species"]["n_s"] / old_state["nB"],
            "new_nu_over_nB": new_state["species"]["n_u"] / new_state["nB"],
            "new_nd_over_nB": new_state["species"]["n_d"] / new_state["nB"],
            "new_ns_over_nB": new_state["species"]["n_s"] / new_state["nB"],
        }
    )

pqm_compare_df = pd.DataFrame(rows)

# Validation checks for the new EOS.
mu_u0, mu_d0, mu_s0 = sqm._quark_mu_triplet_neutral(compare_muB, 0.0, compare_T, ms=compare_ms, upB=compare_upB)
muK0_equal = np.allclose([mu_u0, mu_d0, mu_s0], compare_muB / 3.0, rtol=1.0e-11, atol=1.0e-8)
max_new_charge = float(np.nanmax(np.abs(pqm_compare_df["new_charge_over_3nB"])))
if abs(compare_ms) <= 1.0e-12 and not muK0_equal:
    raise AssertionError("For massless quarks at muK=0, neutral EOS should give mu_u = mu_d = mu_s = muB/3.")
if max_new_charge > 1.0e-9:
    raise AssertionError(f"New EOS charge-neutrality residual is too large: {max_new_charge:.3e}")

summary = pd.DataFrame(
    {
        "quantity": [
            "max |old charge|/(3 nB)",
            "max |new charge|/(3 nB)",
            "P_old at muK=0 [MeV^4]",
            "P_new at muK=0 [MeV^4]",
            "P_old at max muK [MeV^4]",
            "P_new at max muK [MeV^4]",
            "muQ_new at max muK [MeV]",
        ],
        "value": [
            float(np.nanmax(np.abs(pqm_compare_df["old_charge_over_3nB"]))),
            max_new_charge,
            float(pqm_compare_df.loc[pqm_compare_df["muK_MeV"].idxmin(), "P_old_MeV4"]),
            float(pqm_compare_df.loc[pqm_compare_df["muK_MeV"].idxmin(), "P_new_MeV4"]),
            float(pqm_compare_df.loc[pqm_compare_df["muK_MeV"].idxmax(), "P_old_MeV4"]),
            float(pqm_compare_df.loc[pqm_compare_df["muK_MeV"].idxmax(), "P_new_MeV4"]),
            float(pqm_compare_df.loc[pqm_compare_df["muK_MeV"].idxmax(), "muQ_new_MeV"]),
        ],
    }
)

print(
    f"Compared old vs new quark EOS at muB={compare_muB:g} MeV, "
    f"T={compare_T:g} MeV, B^(1/4)={compare_B14:g} MeV, ms={compare_ms:g} MeV."
)
_display(summary)
_display(pqm_compare_df.head())


## Charge Neutral PQM Pressure And Charge Visualization

Contains the `CHARGE_NEUTRAL_PQM_PRESSURE_AND_CHARGE_VISUALIZATION` notebook section.


In [ ]:
# CHARGE_NEUTRAL_PQM_PRESSURE_AND_CHARGE_VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), constrained_layout=True)

ax = axes[0]
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["P_old_MeV4"] / 1.0e9, label="old non-neutral PQM", lw=2.2)
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["P_new_MeV4"] / 1.0e9, label="new neutral PQM", lw=2.2)
ax.set_xlabel(r"$\mu_K$ [MeV]")
ax.set_ylabel(r"$P_{QM}$ [$10^9$ MeV$^4$]")
ax.set_title(rf"Pressure at $\mu_B={compare_muB:g}$ MeV, $T={compare_T:g}$ MeV")
ax.grid(alpha=0.25)
ax.legend(frameon=False)

ax = axes[1]
ax.axhline(0.0, color="0.2", lw=1.0)
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["old_charge_over_3nB"], label="old non-neutral", lw=2.2)
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["new_charge_over_3nB"], label="new neutral", lw=2.2)
ax.set_xlabel(r"$\mu_K$ [MeV]")
ax.set_ylabel(r"$(2n_u - n_d - n_s)/(3n_B)$")
ax.set_title("Bulk electric-charge residual")
ax.grid(alpha=0.25)
ax.legend(frameon=False)

plt.show()


## Charge Neutral PQM Muq And Composition Visualization

Contains the `CHARGE_NEUTRAL_PQM_MUQ_AND_COMPOSITION_VISUALIZATION` notebook section.


In [ ]:
# CHARGE_NEUTRAL_PQM_MUQ_AND_COMPOSITION_VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), constrained_layout=True)

ax = axes[0]
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["muQ_new_MeV"], color="tab:purple", lw=2.2)
ax.axhline(0.0, color="0.2", lw=1.0)
ax.set_xlabel(r"$\mu_K$ [MeV]")
ax.set_ylabel(r"solved auxiliary $\mu_Q$ [MeV]")
ax.set_title(r"Quark-sector charge-neutrality multiplier")
ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["new_nu_over_nB"], label=r"new $n_u/n_B$", lw=2.2)
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["new_nd_over_nB"], label=r"new $n_d/n_B$", lw=2.2)
ax.plot(pqm_compare_df["muK_MeV"], pqm_compare_df["new_ns_over_nB"], label=r"new $n_s/n_B$", lw=2.2)
ax.set_xlabel(r"$\mu_K$ [MeV]")
ax.set_ylabel(r"density fraction")
ax.set_title("New neutral EOS composition")
ax.grid(alpha=0.25)
ax.legend(frameon=False)

plt.show()


## Charge Neutral PQM Old Vs New Composition Visualization

Contains the `CHARGE_NEUTRAL_PQM_OLD_VS_NEW_COMPOSITION_VISUALIZATION` notebook section.


In [ ]:
# CHARGE_NEUTRAL_PQM_OLD_VS_NEW_COMPOSITION_VISUALIZATION
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), sharex=True, constrained_layout=True)
flavors = [("u", "tab:blue"), ("d", "tab:orange"), ("s", "tab:green")]
for ax, (flavor, color) in zip(axes, flavors):
    ax.plot(
        pqm_compare_df["muK_MeV"],
        pqm_compare_df[f"old_n{flavor}_over_nB"],
        ls="--",
        color=color,
        lw=2.0,
        label=f"old n_{flavor}/n_B",
    )
    ax.plot(
        pqm_compare_df["muK_MeV"],
        pqm_compare_df[f"new_n{flavor}_over_nB"],
        ls="-",
        color=color,
        lw=2.2,
        label=f"new n_{flavor}/n_B",
    )
    ax.set_xlabel(r"$\mu_K$ [MeV]")
    ax.set_title(f"{flavor}-quark fraction")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
axes[0].set_ylabel(r"density fraction")
plt.show()


## Analytic Velocity Contour Visualization

Contains the `ANALYTIC_VELOCITY_CONTOUR_VISUALIZATION` notebook section.


In [ ]:
# ANALYTIC_VELOCITY_CONTOUR_VISUALIZATION
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import RMFsolver.constants as const

# Choose which saved analytic mesh to visualize: "linear" or "log".
analytic_visualization_mesh = "linear"

# Set this to None to use the axis scale that matches the selected saved mesh.
# Or set it manually to "linear" or "log".
analytic_temperature_axis_scale = None

def _latest_saved_analytic_path(mesh_label):
    mesh_label = str(mesh_label).lower()
    candidates = [
        path
        for path in Path(".").glob(f"26SU-2-hydro-analytic-velocity-contour-linear_cluster.npy")
        if path.is_file()
    ]
    if not candidates:
        return Path(f"26SU-2-analytic-velocity-contour-{mesh_label}.npy")
    return max(candidates, key=lambda path: path.stat().st_mtime)


analytic_scan_npy_paths = globals().get(
    "analytic_scan_npy_paths",
    {
        "linear": _latest_saved_analytic_path("linear"),
        "log": _latest_saved_analytic_path("log"),
    },
)

# Shared comparison controls. The archived-result plot below reuses these when run after this cell.
comparison_contour_levels_m_s = [50, 100, 200, 500, 1000, 2000]
comparison_use_fixed_level_colors = True
comparison_contour_colors = {
    50.0: "#1f77b4",
    100.0: "#17becf",
    200.0: "#ff7f0e",
    500.0: "#8c6d31",
    1000.0: "#2ca02c",
    2000.0: "#9467bd",
    5000.0: "#d62728",
}
comparison_contour_cmap = "rainbow"
comparison_contour_linewidth = 1.8
comparison_contour_label_fontsize = 11
comparison_axis_fontsize = 15
comparison_title_fontsize = 17
comparison_tick_fontsize = 12
comparison_legend_fontsize = 11
comparison_figure_size = (6.0, 4.2)
comparison_x_limits = (0.01, 0.9)
comparison_y_limits = (0.01, 100.0)
comparison_auto_contour_count = 7

analytic_aQstar_shade_color = "grey"
analytic_aQstar_shade_alpha = 0.6
analytic_aQstar_boundary_color = "0.15"
analytic_aQstar_boundary_linewidth = 1.5
analytic_aQstar_boundary_linestyle = "-"
analytic_aQstar_boundary_label = r"$a_{Q*}^{\rm max}=1$"
analytic_failed_marker_size = 14
analytic_missing_marker_size = 10

analytic_visualization_mesh = str(analytic_visualization_mesh).lower()
if analytic_visualization_mesh not in analytic_scan_npy_paths:
    raise RuntimeError("analytic_visualization_mesh must be 'linear' or 'log'")
analytic_scan_npy_path = Path(analytic_scan_npy_paths[analytic_visualization_mesh])
if not analytic_scan_npy_path.exists():
    raise RuntimeError(f"Run the analytic scan cell first; missing saved file {analytic_scan_npy_path}")

analytic_payload = np.load(analytic_scan_npy_path, allow_pickle=True).item()
analytic_scan_df = pd.DataFrame.from_records(list(analytic_payload.get("records", [])))
if analytic_scan_df.empty:
    raise RuntimeError(f"Saved analytic file {analytic_scan_npy_path} contains no scan records")

n0 = float(analytic_payload.get("n0", globals().get("n0", 0.16 * const.MeV_fm**3)))
nB_crit = float(analytic_payload.get("nB_crit", globals().get("nB_crit", 3.0 * n0)))
B14_for_summary = float(analytic_payload.get("B14", globals().get("B14", np.nan)))
analytic_xi = float(analytic_payload.get("xi", globals().get("analytic_xi", 0.0)))
analytic_temperature_values = np.asarray(analytic_payload.get("temperature_values"), dtype=float)
analytic_metastability_values = np.asarray(analytic_payload.get("metastability_values"), dtype=float)
if analytic_temperature_values.size == 0 or analytic_metastability_values.size == 0:
    raise RuntimeError(f"Saved analytic file {analytic_scan_npy_path} is missing grid values")

if "temperature_mesh" in analytic_scan_df.columns:
    analytic_scan_df = analytic_scan_df[analytic_scan_df["temperature_mesh"].astype(str).str.lower() == analytic_visualization_mesh].copy()
if not analytic_scan_df.empty and "B14_calibrated_MeV" in analytic_scan_df.columns:
    _cached_B14 = pd.to_numeric(analytic_scan_df["B14_calibrated_MeV"], errors="coerce")
    analytic_scan_df = analytic_scan_df[np.isfinite(_cached_B14) & (_cached_B14 > 0.0)].copy()

latest_analytic_df = (
    analytic_scan_df
    .dropna(subset=["T_MeV", "metastability"])
    .drop_duplicates(subset=["T_MeV", "metastability"], keep="last")
)
if latest_analytic_df.empty:
    raise RuntimeError(f"No latest analytic rows are available in {analytic_scan_npy_path}")


def _analytic_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in ("true", "1", "yes")
    return bool(value)


if "endpoint_domain_failure" in latest_analytic_df.columns:
    endpoint_domain_failure = latest_analytic_df["endpoint_domain_failure"].map(_analytic_bool)
else:
    endpoint_domain_failure = pd.Series(False, index=latest_analytic_df.index)
if "message" in latest_analytic_df.columns:
    endpoint_domain_failure = endpoint_domain_failure | latest_analytic_df["message"].astype(str).str.contains(
        "aQstar_max must satisfy 0 < aQstar_max < 1",
        regex=False,
    )
latest_analytic_df = latest_analytic_df.assign(_endpoint_domain_failure=endpoint_domain_failure.to_numpy(dtype=bool))

velocity_pivot = (
    latest_analytic_df.pivot_table(index="T_MeV", columns="metastability", values="u_N_m_s", aggfunc="last")
    .reindex(index=analytic_temperature_values, columns=analytic_metastability_values)
)
success_pivot = (
    latest_analytic_df.pivot_table(index="T_MeV", columns="metastability", values="success", aggfunc="last")
    .reindex(index=analytic_temperature_values, columns=analytic_metastability_values)
)
aQstar_pivot = (
    latest_analytic_df.pivot_table(index="T_MeV", columns="metastability", values="aQstar_max", aggfunc="last")
    .reindex(index=analytic_temperature_values, columns=analytic_metastability_values)
)
endpoint_pivot = (
    latest_analytic_df.pivot_table(index="T_MeV", columns="metastability", values="_endpoint_domain_failure", aggfunc="last")
    .reindex(index=analytic_temperature_values, columns=analytic_metastability_values)
)

missing_grid = success_pivot.isna().to_numpy(dtype=bool)
success_grid = success_pivot.infer_objects(copy=False).fillna(False).astype(bool).to_numpy(dtype=bool)
endpoint_domain_grid = endpoint_pivot.infer_objects(copy=False).fillna(False).astype(bool).to_numpy(dtype=bool)
velocity_grid = velocity_pivot.to_numpy(dtype=float)
aQstar_grid = aQstar_pivot.to_numpy(dtype=float)
velocity_grid[~success_grid] = np.nan

M_grid, T_grid = np.meshgrid(analytic_metastability_values, analytic_temperature_values)
masked_velocity = np.ma.masked_invalid(velocity_grid)
masked_aQstar = np.ma.masked_invalid(aQstar_grid)
finite_velocity = velocity_grid[np.isfinite(velocity_grid) & (velocity_grid > 0.0)]
finite_aQstar = aQstar_grid[np.isfinite(aQstar_grid)]
# Hydro analytic scans may stop at aQstar_max=1 and store the forbidden side
# as endpoint-domain failures rather than finite aQstar_max > 1 diagnostics.
endpoint_forbidden_grid = endpoint_domain_grid | (np.isfinite(aQstar_grid) & (aQstar_grid > 1.0))
endpoint_forbidden_mask = np.ma.masked_where(~endpoint_forbidden_grid, endpoint_forbidden_grid.astype(float))


def _comparison_contour_levels(values, manual_levels=None, target_count=7):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0.0)]
    if values.size < 2:
        return np.array([], dtype=float)
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    if np.isclose(vmin, vmax):
        return np.array([], dtype=float)
    if manual_levels is not None:
        levels = np.asarray(manual_levels, dtype=float)
        levels = levels[np.isfinite(levels) & (levels > 0.0)]
        levels = np.unique(levels)
        return levels[(levels >= vmin) & (levels <= vmax)]
    if vmax / vmin >= 20.0:
        raw = np.geomspace(vmin, vmax, target_count)
    else:
        raw = np.linspace(vmin, vmax, target_count)
    return np.unique(np.asarray(raw, dtype=float))


def _comparison_contour_label(value):
    value = float(value)
    if abs(value) >= 1.0e4:
        exponent = int(np.floor(np.log10(abs(value))))
        mantissa = value / (10.0 ** exponent)
        mantissa_text = f"{mantissa:.1f}".rstrip("0").rstrip(".")
        return rf"${mantissa_text}\times10^{{{exponent}}}$"
    if abs(value) >= 100.0:
        return f"{value:.0f}"
    if abs(value) >= 10.0:
        return f"{value:.1f}".rstrip("0").rstrip(".")
    return f"{value:.2g}"


def _comparison_colors_for_levels(levels, color_map, use_fixed_colors=True):
    if not use_fixed_colors:
        return None
    colors = []
    for level in levels:
        matching_color = None
        for color_level, color in color_map.items():
            if np.isclose(float(level), float(color_level)):
                matching_color = color
                break
        if matching_color is None:
            return None
        colors.append(matching_color)
    return colors


axis_scale = analytic_temperature_axis_scale
if axis_scale is None:
    axis_scale = str(analytic_payload.get("temperature_axis_scale", analytic_visualization_mesh)).lower()
axis_scale = str(axis_scale).lower()
if axis_scale not in ("linear", "log"):
    raise RuntimeError("analytic_temperature_axis_scale must be None, 'linear', or 'log'")
if axis_scale == "log" and comparison_y_limits[0] <= 0.0:
    raise RuntimeError("Log temperature axis requires a positive lower y-limit")
comparison_temperature_axis_scale = axis_scale

fig, ax = plt.subplots(figsize=comparison_figure_size, constrained_layout=True, facecolor="white")
ax.set_facecolor("white")
ax.grid(False)

# Forbidden endpoint-domain region: analytic expansion requires aQstar_max < 1.
aQstar_boundary_drawn = False
if finite_aQstar.size:
    aQstar_min = float(np.nanmin(finite_aQstar))
    aQstar_max = float(np.nanmax(finite_aQstar))
else:
    aQstar_min = np.nan
    aQstar_max = np.nan

if np.any(endpoint_forbidden_grid):
    ax.contourf(
        M_grid,
        T_grid,
        endpoint_forbidden_mask,
        levels=[0.5, 1.5],
        colors=[analytic_aQstar_shade_color],
        alpha=float(analytic_aQstar_shade_alpha),
        zorder=0,
    )

if finite_aQstar.size and aQstar_min <= 1.0 <= aQstar_max:
    ax.contour(
        M_grid,
        T_grid,
        masked_aQstar,
        levels=[1.0],
        colors=[analytic_aQstar_boundary_color],
        linewidths=float(analytic_aQstar_boundary_linewidth),
        linestyles=analytic_aQstar_boundary_linestyle,
        zorder=4,
    )
    aQstar_boundary_drawn = True
elif np.nanmin(endpoint_forbidden_grid.astype(float)) < 0.5 < np.nanmax(endpoint_forbidden_grid.astype(float)):
    ax.contour(
        M_grid,
        T_grid,
        endpoint_forbidden_grid.astype(float),
        levels=[0.5],
        colors=[analytic_aQstar_boundary_color],
        linewidths=float(analytic_aQstar_boundary_linewidth),
        linestyles=analytic_aQstar_boundary_linestyle,
        zorder=4,
    )
    aQstar_boundary_drawn = True

levels = _comparison_contour_levels(
    finite_velocity,
    manual_levels=comparison_contour_levels_m_s,
    target_count=int(comparison_auto_contour_count),
)
if levels.size:
    contour_colors = _comparison_colors_for_levels(
        levels,
        comparison_contour_colors,
        use_fixed_colors=bool(comparison_use_fixed_level_colors),
    )
    contour_kwargs = {
        "levels": levels,
        "linewidths": float(comparison_contour_linewidth),
        "zorder": 3,
    }
    if contour_colors is None:
        contour_kwargs["cmap"] = comparison_contour_cmap
    else:
        contour_kwargs["colors"] = contour_colors
    contour = ax.contour(M_grid, T_grid, masked_velocity, **contour_kwargs)
    label_texts = ax.clabel(
        contour,
        contour.levels,
        inline=True,
        inline_spacing=8,
        fmt={level: _comparison_contour_label(level) for level in contour.levels},
        fontsize=int(comparison_contour_label_fontsize),
    )
    for text in label_texts:
        text.set_bbox({"facecolor": "white", "edgecolor": "none", "alpha": 0.86, "pad": 0.8})
else:
    ax.text(
        0.03,
        0.97,
        "no finite velocity contours in selected levels",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=int(comparison_legend_fontsize),
        color="0.25",
        bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9, "pad": 3.0},
    )

failure_mask = ~success_grid & ~missing_grid & ~endpoint_domain_grid
if np.any(failure_mask):
    ax.scatter(
        M_grid[failure_mask],
        T_grid[failure_mask],
        s=float(analytic_failed_marker_size),
        facecolors="none",
        edgecolors="0.25",
        linewidths=0.8,
        label="latest failed analytic point",
        zorder=5,
    )
if np.any(missing_grid):
    ax.scatter(
        M_grid[missing_grid],
        T_grid[missing_grid],
        s=float(analytic_missing_marker_size),
        marker="x",
        color="0.35",
        linewidths=0.7,
        label="missing analytic point",
        zorder=5,
    )

legend_handles, legend_labels = ax.get_legend_handles_labels()
if aQstar_boundary_drawn and analytic_aQstar_boundary_label:
    legend_handles.append(
        plt.Line2D(
            [0],
            [0],
            color=analytic_aQstar_boundary_color,
            linewidth=float(analytic_aQstar_boundary_linewidth),
            linestyle=analytic_aQstar_boundary_linestyle,
            label=analytic_aQstar_boundary_label,
        )
    )
    legend_labels.append(analytic_aQstar_boundary_label)
if legend_handles:
    ax.legend(legend_handles, legend_labels, loc="best", fontsize=int(comparison_legend_fontsize))

ax.set_xscale("linear")
ax.set_yscale(axis_scale)
ax.set_xlim(*comparison_x_limits)
ax.set_ylim(*comparison_y_limits)
ax.set_xlabel(r"$(n_B^N-n_B^{\rm crit})/n_0$", fontsize=int(comparison_axis_fontsize))
ax.set_ylabel(r"Temperature [MeV]", fontsize=int(comparison_axis_fontsize))
ax.set_title(rf"hydro analytic: $n_B^{{\rm crit}}(T=0) = {float(nB_crit / n0):.1f}\,n_0$", fontsize=int(comparison_title_fontsize))
ax.tick_params(direction="in", top=True, right=True, length=5, width=1.0, labelsize=int(comparison_tick_fontsize))

plt.show()

analytic_summary_table = pd.DataFrame([
    {
        "mesh": analytic_visualization_mesh,
        "axis_scale": axis_scale,
        "source_file": str(analytic_scan_npy_path),
        "B14_calibrated_MeV": float(B14_for_summary),
        "nB_crit_over_n0": float(nB_crit / n0),
        "grid": f"{len(analytic_temperature_values)} x {len(analytic_metastability_values)}",
        "xi": float(analytic_xi),
        "successes": int(success_grid.sum()),
        "endpoint_domain_failures": int(endpoint_domain_grid.sum()),
        "shaded_forbidden_points": int(endpoint_forbidden_grid.sum()),
        "latest_failures": int(failure_mask.sum()),
        "missing": int(missing_grid.sum()),
        "total_points": int(success_grid.size),
        "u_N_min_m_s": float(np.nanmin(finite_velocity)) if finite_velocity.size else np.nan,
        "u_N_max_m_s": float(np.nanmax(finite_velocity)) if finite_velocity.size else np.nan,
        "aQstar_min": float(aQstar_min) if np.isfinite(aQstar_min) else np.nan,
        "aQstar_max": float(aQstar_max) if np.isfinite(aQstar_max) else np.nan,
        "contour_levels_m_s": ", ".join(_comparison_contour_label(level).replace("$", "") for level in levels),
    }
])
display(analytic_summary_table)
